# Step 4: Needle in a Haystack with a quantized KV cache (Qwen3-4B, T4)

**Goal.** Run the same needle grid as the FP16 baseline, but with the K/V values stored through a quantizer, and see whether retrieval drops. The target is an FP8 cache; if FP8 does not lose anything, the notebook escalates to a harsher format so the report can say where degradation begins.

**What is simulated, and what is not**
- Every K and V tensor is rounded to the target format when it is written to the cache. Accuracy is therefore what a real quantized cache would give. The tensors are still kept in FP16, so the memory saving is *not* measured here; it comes from the formula (FP8 = 1 byte per element instead of 2).
- `PREFILL_EXACT = True` (default): the prompt's own attention during prefill uses exact K/V, and only the *stored* cache is quantized, so all decode steps read quantized values. This is how FP8 KV caches in serving engines behave. Set it to `False` for a harsher test where prefill also sees quantized values.
- No scaling is applied for FP8 (values are clamped to the format's range, like an uncalibrated FP8 cache).

**Pairing with the FP16 run.** The prompts are built by exactly the same code and seed as in the FP16 notebook, so each `(length, depth, variant)` trial is the same prompt. **Do not edit `build_prompt`**, otherwise the comparison is no longer paired. The FP16 results file `needle_fp16.jsonl` must be in the Drive results folder `MyDrive/kv-cache/results` (the check cell tells you). Results are kept on Drive because every Colab notebook gets its own runtime and `/content` is not shared.

**Order:** setup, model, haystack, prompt builder, generation, quantizers, cache, smoke test, FP16 file check, run (with escalation), paired comparison, download.

In [ ]:
%pip install -q -U transformers accelerate datasets matplotlib pandas tqdm

In [ ]:
import json, os, random, re, time, math
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "Switch the runtime to a GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print(torch.__version__, GPU_NAME, f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")

# Results live on Google Drive: they survive runtime resets and are shared with the FP8 notebook
# (every notebook gets its own Colab runtime, and /content is not shared between runtimes).
from google.colab import drive
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/kv-cache/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

## Config
`ESCALATION` is tried in order. The first mode runs the full grid; the later ones run only at 16K (where degradation is most likely) to save time. The loop stops at the first mode that is below 100%.

In [ ]:
CFG = dict(
    model_id="Qwen/Qwen3-4B",
    lengths=[2048, 4096, 8192, 16384],
    depths=[0, 25, 50, 75, 100],
    n_variants=5,
    max_new_tokens=24,
    seed=0,                      # must match the FP16 run
    tag="fp8_e4m3",
)
FP16_PATH = f"{RESULTS_DIR}/needle_fp16.jsonl"
SMOKE_PATH = f"{RESULTS_DIR}/needle_smoke_quant.jsonl"
SMOKE = dict(lengths=[2048], depths=[0, 50, 100], n_variants=2)

PREFILL_EXACT = True
ESCALATION = ["fp8_e4m3", "fp8_e5m2", "int4"]     # add "int8" if you want a point between FP8 and INT4
FULL = dict(lengths=CFG["lengths"], depths=CFG["depths"], n_variants=CFG["n_variants"])
ESC  = dict(lengths=[16384], depths=CFG["depths"], n_variants=CFG["n_variants"])

## Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(CFG["model_id"])
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_id"], torch_dtype=torch.float16, device_map="cuda", attn_implementation="sdpa"
).eval()

mc = model.config
head_dim = getattr(mc, "head_dim", mc.hidden_size // mc.num_attention_heads)
KV_BYTES_PER_TOKEN = 2 * mc.num_hidden_layers * mc.num_key_value_heads * head_dim * 2   # FP16
weights_gib = sum(p.numel() * p.element_size() for p in model.parameters()) / 2**30
print(f"layers {mc.num_hidden_layers}, q heads {mc.num_attention_heads}, kv heads {mc.num_key_value_heads}, head_dim {head_dim}")
print(f"weights {weights_gib:.2f} GiB, KV cache {KV_BYTES_PER_TOKEN/1024:.0f} KiB/token "
      f"-> {KV_BYTES_PER_TOKEN*16384/2**30:.2f} GiB at 16K")

# sanity check
msgs = [{"role": "user", "content": "Say hello in five words."}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False,
                              return_tensors="pt", return_dict=True).to("cuda")
out = model.generate(**ids, max_new_tokens=16, do_sample=False, temperature=None, top_p=None, top_k=None)
print(repr(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)))

## Haystack

In [ ]:
from datasets import load_dataset

ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
raw = " ".join(t.strip() for t in ds["text"] if t.strip() and not t.strip().startswith("="))
raw = raw.replace(" @-@ ", "-").replace(" @,@ ", ",").replace(" @.@ ", ".")
SENTS = [s for s in re.split(r"(?<=[.!?])\s+", raw) if 40 <= len(s) <= 400][:40000]
SENT_TOK = [len(x) for x in tok(SENTS, add_special_tokens=False)["input_ids"]]
print(len(SENTS), "sentences,", sum(SENT_TOK), "tokens, mean", round(np.mean(SENT_TOK), 1), "tokens/sentence")

## Needles and prompt builder
Copied unchanged from the FP16 notebook so the prompts are identical.

In [ ]:
PLACES = ["Marlowe Harbor", "Quillfeather", "Brindlemoor", "Tessaly Ridge",
          "Orrin Falls", "Valdemar Bay", "Pellam Crossing", "Ashgrove Mill"]

def ntok(s):
    return len(tok(s, add_special_tokens=False)["input_ids"])

def build_prompt(target_len, depth, variant, seed=CFG["seed"], overhead=60):
    place = PLACES[variant % len(PLACES)]
    value = str(random.Random(f"val-{seed}-{variant}").randint(100000, 999999))
    needle = f"The special magic number for {place} is {value}."
    question = f"What is the special magic number for {place}?"
    budget = target_len - overhead - ntok(needle) - ntok(question)

    rng = random.Random(f"hay-{seed}-{target_len}-{variant}")       # same haystack at every depth
    start = rng.randrange(0, len(SENTS) - 2000)
    chosen, total, i = [], 0, start
    while total < budget:
        chosen.append(SENTS[i]); total += SENT_TOK[i]; i += 1

    k = round(depth / 100 * len(chosen))                             # needle goes before sentence k
    actual_depth = 100 * sum(SENT_TOK[start:start + k]) / total
    text = " ".join(chosen[:k] + [needle] + chosen[k:])
    prompt = f"{text}\n\nBased only on the text above, {question} Reply with just the number."
    return prompt, dict(value=value, actual_depth=actual_depth, place=place)

# check: the value appears exactly once (in the needle), length and depth are close to the targets
p, m = build_prompt(2048, 50, 0)
assert p.count(m["value"]) == 1
print("tokens in prompt text:", ntok(p), "| actual depth:", round(m["actual_depth"], 1), "%")

## Generation
Same as before, except the cache argument is named `cache` in transformers 5.x and `past_key_values` in 4.x.

In [ ]:
import transformers
CACHE_KW = "cache" if int(transformers.__version__.split(".")[0]) >= 5 else "past_key_values"

@torch.no_grad()
def answer(prompt, max_new_tokens, cache=None):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to("cuda")
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.perf_counter()
    kw = {CACHE_KW: cache} if cache is not None else {}
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, top_k=None, **kw)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    peak = torch.cuda.max_memory_allocated()
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True), ids.shape[1], dt, peak

## Trial and grid runner
Same as before: one JSON line per trial, resumes after a disconnect, marks OOM lengths.

In [ ]:
def run_trial(L, d, v, make_cache=None):
    prompt, meta = build_prompt(L, d, v)
    cache = make_cache() if make_cache is not None else None
    out_text, n_in, dt, peak = answer(prompt, CFG["max_new_tokens"], cache)
    correct = meta["value"] in out_text.replace(",", "").replace(" ", "")
    return dict(tag=CFG["tag"], length=L, depth=d, variant=v, n_tokens=n_in,
                actual_depth=round(meta["actual_depth"], 1), expected=meta["value"],
                output=out_text.strip(), correct=bool(correct),
                seconds=round(dt, 2), peak_gib=round(peak / 2**30, 2), status="ok")

def run_grid(lengths, depths, n_variants, path, make_cache=None, retry_oom=False):
    done = set()
    if os.path.exists(path):
        for line in open(path):
            r = json.loads(line)
            if r["status"] == "ok" or not retry_oom:
                done.add((r["length"], r["depth"], r["variant"]))
    todo = [(L, d, v) for L in lengths for d in depths for v in range(n_variants) if (L, d, v) not in done]
    oom_at = []
    with open(path, "a") as f:
        for L, d, v in tqdm(todo):
            if any(L >= x for x in oom_at):
                rec = dict(tag=CFG["tag"], length=L, depth=d, variant=v, status="oom")
            else:
                try:
                    rec = run_trial(L, d, v, make_cache)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache(); oom_at.append(L)
                    print(f"OOM at length {L}; marking this and longer lengths as oom")
                    rec = dict(tag=CFG["tag"], length=L, depth=d, variant=v, status="oom")
            f.write(json.dumps(rec) + "\n"); f.flush()
    return pd.read_json(path, lines=True)

## Quantizers
- **FP8 e4m3 / e5m2:** round-trip through the 8-bit float format (values clamped to the format's range). e4m3 has 3 mantissa bits (relative error up to about 6%), e5m2 has 2 (about 12%), with a larger range. If the GPU or PyTorch build cannot cast to float8, an exact emulation is used instead. The first cell below checks that the native cast and the emulation agree.
- **INT8 / INT4:** asymmetric min-max quantization per token and per head (over `head_dim`). This is a simple baseline; methods like KIVI quantize keys per channel because keys have outlier channels, so per-token INT4 on keys is expected to be harmful.

In [ ]:
FP8_FMT = {"fp8_e4m3": dict(dtype="float8_e4m3fn", mb=3, emin=-6, vmax=448.0),
           "fp8_e5m2": dict(dtype="float8_e5m2",   mb=2, emin=-14, vmax=57344.0)}

def fp8_native(x, fmt):
    c = FP8_FMT[fmt]
    return x.float().clamp(-c["vmax"], c["vmax"]).to(getattr(torch, c["dtype"])).to(x.dtype)

def fp8_emulated(x, fmt):
    c = FP8_FMT[fmt]
    xf = x.float().clamp(-c["vmax"], c["vmax"])
    e = torch.floor(torch.log2(xf.abs().clamp(min=2.0 ** (c["emin"] - c["mb"]))))
    e = e.clamp(min=c["emin"])
    step = 2.0 ** (e - c["mb"])
    return (torch.round(xf / step) * step).clamp(-c["vmax"], c["vmax"]).to(x.dtype)

def quant_int(x, bits):
    xf = x.float()
    mn, mx = xf.amin(-1, keepdim=True), xf.amax(-1, keepdim=True)
    qmax = 2 ** bits - 1
    scale = ((mx - mn) / qmax).clamp(min=1e-8)
    return (torch.round((xf - mn) / scale).clamp(0, qmax) * scale + mn).to(x.dtype)

# choose native or emulated FP8 for this GPU
probe = torch.cat([torch.randn(1 << 15), torch.randn(1 << 15) * 1e-3, torch.randn(1 << 15) * 100]).half().cuda()
IMPL = {}
for fmt in FP8_FMT:
    emu = fp8_emulated(probe, fmt)
    try:
        nat = fp8_native(probe, fmt)
        IMPL[fmt] = fp8_native
        print(f"{fmt}: native cast works; differs from emulation in {int((nat != emu).sum())} of {probe.numel()} values")
    except Exception as e:
        IMPL[fmt] = fp8_emulated
        print(f"{fmt}: native cast failed ({type(e).__name__}); using emulation")

def get_qfn(mode):
    if mode in FP8_FMT:
        f = IMPL[mode]; return lambda x: f(x, mode)
    if mode == "int8": return lambda x: quant_int(x, 8)
    if mode == "int4": return lambda x: quant_int(x, 4)
    raise ValueError(mode)

# relative error on random data, as a rough scale of how lossy each mode is
x = torch.randn(1, 8, 512, 128, device="cuda").half()
for mode in ["fp8_e4m3", "fp8_e5m2", "int8", "int4"]:
    q = get_qfn(mode)(x)
    print(f"{mode:9s} relative error {((q.float() - x.float()).norm() / x.float().norm()).item():.4f}")

## Quantizing cache
Subclass of `DynamicCache`: every K/V written to the cache goes through the quantizer. With `PREFILL_EXACT`, the first write of each layer (the prompt) returns the exact states for the attention of that forward pass but stores the quantized ones, so every later decode step reads quantized values. The sanity cell prints the real K/V relative error for each mode on an actual prompt.

In [ ]:
from transformers import DynamicCache

class QuantCache(DynamicCache):
    def __init__(self, qfn, prefill_exact=True, record=False):
        try:
            super().__init__(config=model.config)
        except TypeError:
            super().__init__()
        self.qfn, self.prefill_exact = qfn, prefill_exact
        self.stats = [] if record else None

    def update(self, key_states, value_states, layer_idx, *args, **kwargs):
        first = self.get_seq_length(layer_idx) == 0
        qk, qv = self.qfn(key_states), self.qfn(value_states)
        if self.stats is not None:
            kf, vf = key_states.float(), value_states.float()
            self.stats.append((((qk.float() - kf).norm() / kf.norm()).item(),
                               ((qv.float() - vf).norm() / vf.norm()).item()))
        k, v = super().update(qk, qv, layer_idx, *args, **kwargs)
        if first and self.prefill_exact:
            return key_states, value_states
        return k, v

def qcache(mode, record=False):
    return lambda: QuantCache(get_qfn(mode), PREFILL_EXACT, record)

# sanity: run one real prompt per mode and print the K/V error actually seen
prompt, meta = build_prompt(1024, 50, 0)
for mode in ["fp8_e4m3", "fp8_e5m2", "int8", "int4"]:
    c = qcache(mode, record=True)()
    out, n_in, dt, peak = answer(prompt, 8, c)
    s = np.array(c.stats)
    print(f"{mode:9s} K rel. error {s[:, 0].mean():.4f}, V rel. error {s[:, 1].mean():.4f} | answer: {out.strip()!r} (expected {meta['value']})")

## Smoke test
Six trials with the first mode. Check that answers look right and that the time per trial is reasonable.

In [ ]:
CFG["tag"] = ESCALATION[0]
sm = run_grid(SMOKE["lengths"], SMOKE["depths"], SMOKE["n_variants"], SMOKE_PATH, make_cache=qcache(ESCALATION[0]))
print(sm[["length", "n_tokens", "depth", "variant", "expected", "output", "correct", "seconds", "peak_gib"]].to_string())

## FP16 results (needed for the paired comparison)

In [ ]:
if os.path.exists(FP16_PATH):
    n = sum(1 for _ in open(FP16_PATH))
    print(f"found {FP16_PATH}: {n} trials")
else:
    print(f"FP16 results not found at {FP16_PATH}.")
    print("Copy needle_fp16.jsonl from the FP16 run into that Drive folder, then rerun this cell.")

## Run
First mode: the full grid (same layout as the FP16 heatmap). If it is not 100%, stop: that is the result. If it is 100%, the next mode runs at 16K only, and so on. Each full 16K column takes about 10 to 15 minutes. Results go to `needle_<mode>.jsonl`; rerunning resumes.

In [ ]:
def run_mode(mode, grid):
    CFG["tag"] = mode
    path = f"{RESULTS_DIR}/needle_{mode}.jsonl"
    return run_grid(grid["lengths"], grid["depths"], grid["n_variants"], path, make_cache=qcache(mode))

for i, mode in enumerate(ESCALATION):
    d = run_mode(mode, FULL if i == 0 else ESC)
    ok = d[d.status == "ok"]
    print(f"{mode}: {int(ok.correct.sum())}/{len(ok)} correct, {int((d.status == 'oom').sum())} oom")
    if len(ok) and ok.correct.mean() < 1.0:
        print("below 100%: stopping here")
        break

## Results and paired comparison
For each mode: the heatmap, and a per-length table with the FP16 and quantized success counts on the *same* trials. `fp16_only` = FP16 right and quantized wrong; `quant_only` = the reverse. Wrong answers are listed so you can see whether they are garbled or just a different number.

In [ ]:
def summarize(df):
    ok = df[df.status == "ok"]
    return ok.groupby(["length", "depth"])["correct"].agg(k="sum", n="size").reset_index()

def plot_heatmap(df, title, fname):
    g = summarize(df)
    lengths, depths = sorted(g.length.unique()), sorted(g.depth.unique())
    M = np.full((len(depths), len(lengths)), np.nan)
    for r in g.itertuples():
        M[depths.index(r.depth), lengths.index(r.length)] = r.k / r.n
    fig, ax = plt.subplots(figsize=(1.5 * len(lengths) + 2, 0.7 * len(depths) + 1.8))
    im = ax.imshow(M, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(lengths))); ax.set_xticklabels([f"{L//1024}K" for L in lengths])
    ax.set_yticks(range(len(depths)));  ax.set_yticklabels([f"{d}%" for d in depths])
    ax.set_xlabel("context length (tokens)"); ax.set_ylabel("needle depth")
    for r in g.itertuples():
        ax.text(lengths.index(r.length), depths.index(r.depth), f"{int(r.k)}/{int(r.n)}",
                ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, label="success rate")
    ax.set_title(title, fontsize=10)
    fig.savefig(fname, dpi=160, bbox_inches="tight"); plt.show()

In [ ]:
results = {}
for mode in ESCALATION:
    p = f"{RESULTS_DIR}/needle_{mode}.jsonl"
    if os.path.exists(p):
        results[mode] = pd.read_json(p, lines=True)

f16 = pd.read_json(FP16_PATH, lines=True)
f16 = f16[f16.status == "ok"][["length", "depth", "variant", "correct"]].rename(columns={"correct": "fp16"})

rows = []
for mode, d in results.items():
    plot_heatmap(d, f"Needle in a haystack, {mode} cache, {CFG['model_id']}, {GPU_NAME}", f"{RESULTS_DIR}/needle_{mode}_heatmap.png")
    q = d[d.status == "ok"].rename(columns={"correct": "quant"})
    m = q.merge(f16, on=["length", "depth", "variant"], how="inner")
    for L, g in m.groupby("length"):
        rows.append(dict(mode=mode, length=L, n=len(g), fp16=int(g.fp16.sum()), quant=int(g.quant.sum()),
                         fp16_only=int((g.fp16 & ~g.quant).sum()), quant_only=int((~g.fp16 & g.quant).sum())))
    bad = q[~q.quant]
    if len(bad):
        print(f"\n{mode}: wrong answers")
        print(bad[["length", "depth", "variant", "expected", "output"]].to_string(index=False))
print()
print(pd.DataFrame(rows).to_string(index=False))

## Download (optional)
Results are already on Drive. This also downloads a zip and flushes pending Drive writes.

In [ ]:
import shutil
shutil.make_archive("/content/needle_quant_results", "zip", RESULTS_DIR)
try:
    from google.colab import files
    files.download("/content/needle_quant_results.zip")
    drive.flush_and_unmount()
except ImportError:
    print("Not on Colab; results are in", RESULTS_DIR)

## For the report
- State what was simulated: K and V rounded to the target format at write time, stored in FP16; prefill attention exact (`PREFILL_EXACT`), decode reads quantized values. The memory saving comes from the formula, not from a measurement.
- State the GPU, model, FP16 baseline (100 trials, all correct), trials per cell (5) and the prompt setup. A cell with 5 trials has a standard error of about 0.1 to 0.2, so one wrong answer in a cell is weak evidence; look at the table of paired counts and at the total.
- If e4m3 is 100%, say so: "no measurable degradation up to 16K with simulated FP8 e4m3", then report the first mode that does degrade and where.
- Using `PREFILL_EXACT = False` makes the quantizer affect the prompt's own computation as well and is a harsher test; report it separately if you use it.